<a href="https://colab.research.google.com/github/selinb75/AI-Resume-Analyzer./blob/main/AI_Resume_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("Resume.csv")

df.head()

Saving Resume.csv to Resume.csv


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [ ]:
df.columns

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')

In [ ]:
print(df['Category'].value_counts().to_string())


Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22


## Text Cleaning / Text Preprocessing



In [ ]:
!pip install nltk


In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt') #Verileri parçala
nltk.download('stopwords')   # Gereksiz kelimeleri at
nltk.download('wordnet') # Kelimeleri sadeleştir

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:

df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [ ]:
lemmatizer = WordNetLemmatizer() #Akıllı kök bulma ,  Bu fonksiyon kelimenin dil bilgisindeki anlamına bakar.

stop_words = set(stopwords.words('english')) #İngilizcedeki en yaygın ve tek başına anlamsal bir değeri olmayan kelimelerin listesidir. Kümeleme set() daha pratikdir.

def clean_resume(Text):


    Text = Text.lower() # Kelimeleri küçük harfe çevir

    Text = re.sub(r'http\S+', '', Text)     # Linkleri sil

    # Noktalama ve sayıları sil
    Text = re.sub(r'[^a-zA-Z\s]', '',Text) # Sadece İngiliz alfabesini al

    # Tokenization:Temizlenen metni kelimelere ayır
    Words = word_tokenize(Text)

    # Stopwords kaldır + Lemmatization
    cleaned_words = [] #Temizlenen kelimeleri boş listeye alalım

    for word in Words:  #Words : sql python olabilir word döngüsünün içine at
        if word not in stop_words:  #Elimizdeki Kelimeler (word), etkisiz kelimeler kümesinde (stop_words) yoksa
            word = lemmatizer.lemmatize(word) # Kelimeyi al akıllı sözcüğe (Lemattizer) gönderir
            # sözlük kelimeyi köküne (yalın haline) indirger ve biz o yeni kök halini yine aynı word değişkeninin üzerine yazar günceller sırayla.
            cleaned_words.append(word) # kelimeleri listeye at

    return " ".join(cleaned_words)

In [ ]:
nltk.download('punkt_tab')
df["cleaned_Resume"] = df["Resume_str"].apply(clean_resume) #Temizlenmiş texti oluşturalım

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df[["Resume_str", "cleaned_Resume"]].head()

,Resume_str,cleaned_Resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administratormarketing associate hr adminis...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist u hr operation summary versatile...
2,HR DIRECTOR Summary Over 2...,hr director summary year experience recruiting...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,hr manager skill highlight hr skill hr departm...


TF (Term Frequency - Kelime Sıklığı)

Bir kelimenin, incelediğin tek bir özgeçmiş içinde ne kadar sık geçtiğini ölçer.
TF=Kelimenin  cv deki sayısı / Cv deki kelime sayısı
IDF (Inverse Document Frequency - Ters Doküman Sıklığı)
Bir kelimenin tüm veri setinde ne kadar nadir veya ne kadar yaygın olduğunu ölçer. Kelimenin "özel" olup olmadığını anlar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer #Metinleri yapay zekanın anlayabileceği sayılara dönüştürme kütüphanesi
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)
) #en önemli 5000 kelimeyi kullan
X = tfidf.fit_transform(df["cleaned_Resume"]) #Textleri sayısala çevirelim
#fit() fonksiyonu sayesinde veri setinde geçen tüm benzersiz kelimeleri bulur ve devasa bir sözlük listesi çıkarır.
#Ayrıca her kelimenin tüm özgeçmişler içinde ne kadar nadir geçtiğini (yani IDF skorlarını) hesaplar ve bir kenara yazar.
#Transform() Fonksiyonu sayesinde bu devasa sözlüğü 0 ve 1 gibi sayısal verilere dönüştürür.

In [ ]:
X.shape #sonuç olarak 2484 CV 5000 özellik sayısal veriler dönüştü

(2484, 5000)

In [ ]:
print(X[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 329 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 2121)	0.12292342575064048
  (0, 338)	0.056293647576350594
  (0, 133)	0.05936759073913024
  (0, 4460)	0.012180896580307443
  (0, 1175)	0.02917058171237248
  (0, 1084)	0.18570235126691687
  (0, 4079)	0.12277303231583345
  (0, 2673)	0.0555547588883165
  (0, 4982)	0.014320083396406974
  (0, 1661)	0.019821351241449993
  (0, 2106)	0.0839360432527809
  (0, 2632)	0.06715924108816886
  (0, 523)	0.04379290528358652
  (0, 2469)	0.022968735584763523
  (0, 4535)	0.024081115222254035
  (0, 4122)	0.040154611048271074
  (0, 1554)	0.03562230879257896
  (0, 781)	0.037947863886382424
  (0, 2070)	0.01745817124885517
  (0, 1832)	0.028517042128642065
  (0, 3974)	0.025009142430943238
  (0, 2716)	0.21825996738545825
  (0, 3980)	0.045080564770320845
  (0, 893)	0.03167416872730716
  (0, 3817)	0.07785351516950031
  :	:
  (0, 4661)	0.04463196659218504
  (0, 2717)	0.09386198627206946
  (0, 3

CV Category Prediction

In [ ]:
y = df["Category"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
model = LinearSVC()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy_score(y_test, predictions)

0.7243460764587525

In [ ]:
predictions = model.predict(X_test) #tahmini veri

JOB MATCHING SYSTEM

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity  #Benzerlik fonksiyonu (cv benzerliği)
job_description = """
Looking for a Python developer with Django, SQL,
Machine Learning and API development experience.
"""
cleaned_job = clean_resume(job_description)
all_texts = df["cleaned_Resume"].tolist()

all_texts.append(cleaned_job)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000, #TF-IDF skoru en yüksek (yani en ayırt edici, en önemli) olan ilk 5000 kelimeyi seç.
    stop_words='english' #(etkisiz kelimeler) listesini çıkar (the,one)
)

vectors = tfidf.fit_transform(all_texts)

In [ ]:
similarity = cosine_similarity(
    vectors[-1],    #baştan sona cvleri al
    vectors[:-1]
)
best_match = similarity.argmax()

print("Best Match Index:", best_match)

Best Match Index: 926


In [ ]:
print(similarity[0][best_match])

0.21583775241728273


In [ ]:
import numpy as np

top_5 = np.argsort(similarity[0])[-5:]

print(top_5)

[1142 1218 1762  120  926]


In [ ]:
top_5 = np.argsort(similarity[0])[-5:]

for index in reversed(top_5):
    print("CV Index:", index)
    print("Category:", df.iloc[index]["Category"])
    print("\nResume:\n")
    print(df.iloc[index]["Resume_str"][:1000])  # ilk 1000 karakter
    print("\nSimilarity Score:", similarity[0][index])
    print("\n" + "="*80 + "\n")

CV Index: 926
Category: AGRICULTURE

Resume:

           SOFTWARE DEVELOPER         Professional Summary    Enthusiastic computer engineer eager to contribute to team success through hard work, attention to detail and excellent organizational skills. Technical professional with complete understanding of entire software development life cycle. Respectful self-motivator gifted at finding reliable solutions for software issues. Experienced in c#, python, HTML, SQL, node.js/javascript and working knowledge of Restful API design & implementations. Fluent in English and Turkish and accustomed to working with cross-cultural, global teams.      Skills          C#, HTML, CSS, JavaScript, 5 years of experience  SQL, 5 years of experience  Python, MatLab, MongoDB, Tableau, Node JS  Frameworks: .Net, Devexpress, TensorFlow, Keras, Scikit-learn, Pandas, NLTK.  Search Engine Optimization  Net  API  CSS  Clients  Database development  Designing  English  HTML      Image processing  JavaScript  Leader

In [ ]:
generic_phrases = [
    "hardworking",
    "team player",
    "fast learner",
    "passionate",
    "detail oriented",
    "self motivated",
    "excellent communication skills",
    "problem solving",
    "go getter"
]

In [ ]:
def zayıf_cv(text):

    text = text.lower()

    found_phrases = []

    for phrase in generic_phrases:
        if phrase in text:
            found_phrases.append(phrase)

    return found_phrases

In [ ]:
sample_resume = """
I am a hardworking and passionate developer
with excellent communication skills and
problem solving abilities.
"""

result = zayıf_cv(sample_resume)

print(result)

['hardworking', 'passionate', 'excellent communication skills', 'problem solving']


In [ ]:
for i in range(5):

    resume = df.iloc[i]["Resume_str"]

    weak_phrases = zayıf_cv(resume)

    print("CV", i)

    if weak_phrases:
        print("Weak Phrases Found:")
        print(weak_phrases)
    else:
        print("No generic phrases found.")

    print("\n" + "="*50 + "\n")

CV 0
No generic phrases found.


CV 1
No generic phrases found.


CV 2
No generic phrases found.


CV 3
Weak Phrases Found:
['team player']


CV 4
Weak Phrases Found:
['problem solving']




In [ ]:
technical_skills = [
    "python",
    "sql",
    "django",
    "machine learning",
    "tensorflow",
    "java",
    "react",
    "docker",
    "aws"
]

In [ ]:
def count_technical_skills(text):

    text = text.lower()

    count = 0

    for skill in technical_skills:
        if skill in text:
            count += 1

    return count

In [ ]:
resume = df.iloc[0]["Resume_str"]

weak_phrases = zayıf_cv(resume)

skill_count = count_technical_skills(resume)

print("Weak Phrase Count:", len(weak_phrases))
print("Technical Skill Count:", skill_count)

Weak Phrase Count: 0
Technical Skill Count: 1
